# Agentic AI Chatbot - Multi-Agent Document Analysis

This notebook uses an agentic architecture to analyze documents in a Dataiku managed folder.
An orchestrator agent plans analysis, spawns specialized sub-agents, discovers cross-document
connections, and compiles comprehensive answers.

**How it differs from the RAG notebook:**
- Instead of chunking + embedding + vector retrieval, this uses an LLM-driven orchestrator that reasons about *which* documents to examine and *what* to look for
- Sub-agents analyze individual documents, find cross-document connections, and synthesize answers
- The orchestrator operates in a plan-act-observe loop, iteratively refining its understanding

## 1. Configuration

In [ ]:
import dataiku
import os
import json
import re
import io
import numpy as np
from pathlib import PurePosixPath

# ---------------------------------------------------------------------------
# Configuration - edit these or set via Dataiku project variables
# ---------------------------------------------------------------------------
project_variables = dataiku.get_custom_variables()

# Managed folder ID that contains the documents to index
FOLDER_ID = project_variables.get("RAG_FOLDER_ID", "documents")

# LLM configuration (Dataiku LLM Mesh)
LLM_ID = project_variables.get("RAG_LLM_ID", "openai:gpt-4o-mini")

# Agent-specific LLM configuration
AGENT_LLM_ID = project_variables.get("AGENT_LLM_ID", LLM_ID)        # Orchestrator model
SUB_AGENT_LLM_ID = project_variables.get("SUB_AGENT_LLM_ID", LLM_ID)  # Sub-agent model

# Agent parameters
MAX_ITERATIONS = int(project_variables.get("AGENT_MAX_ITERATIONS", 5))
MAX_DOCUMENT_CHARS = int(project_variables.get("AGENT_MAX_DOC_CHARS", 15000))

# Supported file extensions
SUPPORTED_EXTENSIONS = {".txt", ".md", ".csv", ".pdf", ".docx", ".json", ".html"}

# Dataiku project handle (used by all LLM calls)
client = dataiku.api_client()
project = client.get_default_project()

print(f"Folder ID:          {FOLDER_ID}")
print(f"Orchestrator LLM:   {AGENT_LLM_ID}")
print(f"Sub-Agent LLM:      {SUB_AGENT_LLM_ID}")
print(f"Max iterations:     {MAX_ITERATIONS}")
print(f"Max doc chars:      {MAX_DOCUMENT_CHARS}")

## 2. Document Scanning

Recursively scan the Dataiku managed folder and extract text from supported file types.

In [ ]:
# ---------------------------------------------------------------------------
# Text extraction helpers
# ---------------------------------------------------------------------------

def extract_text_txt(raw_bytes):
    """Extract text from plain text / markdown / csv files."""
    return raw_bytes.decode("utf-8", errors="replace")


def extract_text_pdf(raw_bytes):
    """Extract text from a PDF using PyPDF2."""
    try:
        import PyPDF2
    except ImportError:
        raise ImportError("PyPDF2 is required for PDF support. Install it with: pip install PyPDF2")
    reader = PyPDF2.PdfReader(io.BytesIO(raw_bytes))
    pages = [page.extract_text() or "" for page in reader.pages]
    return "\n\n".join(pages)


def extract_text_docx(raw_bytes):
    """Extract text from a DOCX file using python-docx."""
    try:
        import docx
    except ImportError:
        raise ImportError("python-docx is required for DOCX support. Install it with: pip install python-docx")
    doc = docx.Document(io.BytesIO(raw_bytes))
    return "\n\n".join(p.text for p in doc.paragraphs if p.text.strip())


def extract_text_json(raw_bytes):
    """Pretty-print JSON content as text."""
    data = json.loads(raw_bytes.decode("utf-8", errors="replace"))
    return json.dumps(data, indent=2, ensure_ascii=False)


def extract_text_html(raw_bytes):
    """Strip HTML tags and return plain text."""
    text = raw_bytes.decode("utf-8", errors="replace")
    text = re.sub(r"<script[^>]*>[\s\S]*?</script>", "", text, flags=re.IGNORECASE)
    text = re.sub(r"<style[^>]*>[\s\S]*?</style>", "", text, flags=re.IGNORECASE)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


EXTRACTORS = {
    ".txt": extract_text_txt,
    ".md": extract_text_txt,
    ".csv": extract_text_txt,
    ".pdf": extract_text_pdf,
    ".docx": extract_text_docx,
    ".json": extract_text_json,
    ".html": extract_text_html,
}

In [ ]:
# ---------------------------------------------------------------------------
# Scan the managed folder and build document registry
# ---------------------------------------------------------------------------
folder = dataiku.Folder(FOLDER_ID)
all_paths = folder.list_paths_in_partition()

documents = []   # list of {"id": int, "path": str, "ext": str, "text": str, "size": int, "summary": None}
doc_index = {}   # path -> document dict, for fast lookup
skipped = []

for path in sorted(all_paths):
    ext = PurePosixPath(path).suffix.lower()
    if ext not in SUPPORTED_EXTENSIONS:
        skipped.append((path, "unsupported type"))
        continue
    try:
        with folder.get_download_stream(path) as stream:
            raw_bytes = stream.read()
        text = EXTRACTORS[ext](raw_bytes)
        if not text.strip():
            skipped.append((path, "empty content"))
            continue
        doc = {
            "id": len(documents),
            "path": path,
            "ext": ext,
            "text": text,
            "size": len(raw_bytes),
            "summary": None,  # populated lazily by summarizer sub-agent
        }
        documents.append(doc)
        doc_index[path] = doc
    except Exception as e:
        skipped.append((path, str(e)))

# Build a compact manifest for the orchestrator
document_manifest = "\n".join(
    f"- [{doc['id']}] {doc['path']} ({doc['ext']}, {doc['size']:,} bytes, ~{len(doc['text']):,} chars)"
    for doc in documents
)

print(f"Loaded {len(documents)} documents, skipped {len(skipped)} files.")
print()
print(f"{'Path':<60} {'Type':<8} {'Size':>10} {'Text Len':>10}")
print("-" * 90)
for doc in documents:
    print(f"{doc['path']:<60} {doc['ext']:<8} {doc['size']:>10,} {len(doc['text']):>10,}")

if skipped:
    print(f"\nSkipped files:")
    for path, reason in skipped:
        print(f"  {path}: {reason}")

print(f"\nDocument manifest for orchestrator:\n{document_manifest}")

## 3. Sub-Agent Definitions

Each sub-agent is a prompt-driven LLM call through the Dataiku LLM Mesh. No external frameworks are used.

In [ ]:
# ---------------------------------------------------------------------------
# LLM call helpers
# ---------------------------------------------------------------------------

def llm_call(system_prompt, user_prompt, llm_id=None, conversation=None):
    """Make a single LLM call via Dataiku LLM Mesh. Returns the response text."""
    llm_id = llm_id or SUB_AGENT_LLM_ID
    llm = project.get_llm(llm_id)
    completion = llm.new_completion()
    completion.with_message(system_prompt, role="system")
    if conversation:
        for msg in conversation:
            completion.with_message(msg["content"], role=msg["role"])
    completion.with_message(user_prompt, role="user")
    resp = completion.execute()
    return resp.text


def parse_agent_json(text):
    """Extract JSON from an LLM response that may contain markdown fences."""
    match = re.search(r'```(?:json)?\s*([\s\S]*?)```', text)
    if match:
        text = match.group(1)
    return json.loads(text.strip())


print("LLM helpers ready.")

In [ ]:
# ---------------------------------------------------------------------------
# Sub-Agent: Document Analyzer
# ---------------------------------------------------------------------------

ANALYZER_SYSTEM_PROMPT = """You are a document analyst sub-agent. You receive a document and a specific
analysis question. Your job is to carefully read the document and answer the question about it.

You MUST respond with valid JSON in this exact format:
{
    "findings": "your detailed findings from the document",
    "key_entities": ["entity1", "entity2"],
    "key_facts": ["fact1", "fact2"],
    "relevant": true
}

- "findings": A detailed text answer to the analysis question based on the document content.
- "key_entities": Important named entities (people, places, organizations, dates, metrics) found.
- "key_facts": Key factual claims or data points from the document relevant to the question.
- "relevant": true if the document contains relevant information, false otherwise.

Be thorough but concise. Extract specific data points and quotes where possible."""


def agent_analyze_document(doc_path, analysis_question):
    """Analyze a specific document to answer a targeted question about it."""
    if doc_path not in doc_index:
        return {
            "doc_path": doc_path,
            "question": analysis_question,
            "findings": f"Document not found: {doc_path}",
            "key_entities": [],
            "key_facts": [],
            "relevant": False,
        }

    doc = doc_index[doc_path]
    doc_text = doc["text"][:MAX_DOCUMENT_CHARS]

    user_prompt = f"""Document path: {doc_path}
Document content (first {MAX_DOCUMENT_CHARS} chars):
---
{doc_text}
---

Analysis question: {analysis_question}"""

    raw = llm_call(ANALYZER_SYSTEM_PROMPT, user_prompt)
    try:
        result = parse_agent_json(raw)
    except (json.JSONDecodeError, ValueError):
        result = {
            "findings": raw,
            "key_entities": [],
            "key_facts": [],
            "relevant": True,
        }

    result["doc_path"] = doc_path
    result["question"] = analysis_question
    return result


print("Document Analyzer sub-agent ready.")

In [ ]:
# ---------------------------------------------------------------------------
# Sub-Agent: Document Summarizer
# ---------------------------------------------------------------------------

SUMMARIZER_SYSTEM_PROMPT = """You are a document summarizer sub-agent. Produce a concise structured summary
of the provided document. Include:
- What type of document it is (report, dataset, memo, etc.)
- The main topics or subjects covered
- Key data points, metrics, or conclusions
- Notable entities (people, organizations, dates, locations)

Keep the summary under 300 words. Be factual and specific."""


def agent_summarize_document(doc_path):
    """Generate a structured summary of a document. Results are cached."""
    if doc_path not in doc_index:
        return f"Document not found: {doc_path}"

    doc = doc_index[doc_path]

    # Return cached summary if available
    if doc["summary"] is not None:
        return doc["summary"]

    doc_text = doc["text"][:MAX_DOCUMENT_CHARS]
    user_prompt = f"""Document path: {doc_path}
Document content (first {MAX_DOCUMENT_CHARS} chars):
---
{doc_text}
---

Please summarize this document."""

    summary = llm_call(SUMMARIZER_SYSTEM_PROMPT, user_prompt)
    doc["summary"] = summary
    return summary


print("Document Summarizer sub-agent ready.")

In [ ]:
# ---------------------------------------------------------------------------
# Sub-Agent: Connection Finder
# ---------------------------------------------------------------------------

CONNECTION_FINDER_SYSTEM_PROMPT = """You are a cross-document connection finder sub-agent. You receive
findings from analyses of multiple documents and the original user query. Your job is to identify
connections, patterns, and relationships across the documents.

Look for:
- Shared entities: People, organizations, dates, metrics appearing across documents
- Causal links: Where one document's facts explain or are explained by another's
- Contradictions: Conflicting data or claims between documents
- Complementary details: Where documents fill in each other's gaps
- Temporal relationships: How events across documents relate chronologically

You MUST respond with valid JSON in this exact format:
{
    "connections": [
        {
            "type": "shared_entity | causal | contradiction | complementary | temporal",
            "documents": ["/path1", "/path2"],
            "description": "explanation of the connection",
            "significance": "why this matters for the query"
        }
    ],
    "synthesis": "integrated narrative combining findings across documents"
}"""


def agent_find_connections(findings_list, user_query):
    """Analyze findings from multiple documents to discover cross-document connections."""
    findings_text = ""
    for i, f in enumerate(findings_list, 1):
        findings_text += f"""
--- Findings {i} ---
Document: {f.get('doc_path', 'unknown')}
Question asked: {f.get('question', 'N/A')}
Findings: {f.get('findings', 'N/A')}
Key entities: {json.dumps(f.get('key_entities', []))}
Key facts: {json.dumps(f.get('key_facts', []))}
"""

    user_prompt = f"""Original user query: {user_query}

Findings from document analyses:
{findings_text}

Identify all cross-document connections relevant to the user's query."""

    raw = llm_call(CONNECTION_FINDER_SYSTEM_PROMPT, user_prompt)
    try:
        result = parse_agent_json(raw)
    except (json.JSONDecodeError, ValueError):
        result = {
            "connections": [],
            "synthesis": raw,
        }
    return result


print("Connection Finder sub-agent ready.")

In [ ]:
# ---------------------------------------------------------------------------
# Sub-Agent: Answer Synthesizer
# ---------------------------------------------------------------------------

SYNTHESIZER_SYSTEM_PROMPT = """You are an answer synthesis sub-agent. You compile findings from document
analyses and cross-document connections into a coherent, well-structured answer for the user.

Guidelines:
1. Lead with a direct answer to the question
2. Support claims with citations to specific documents (use the document path)
3. Highlight cross-document connections that are relevant to the answer
4. Note any contradictions found between documents
5. Acknowledge gaps where the available documents lack information
6. Keep the tone conversational but precise

Do NOT wrap your response in JSON. Write a natural-language answer."""


def agent_synthesize_answer(user_query, findings, connections, conversation_history=None):
    """Compile all findings and connections into a coherent final answer."""
    findings_text = ""
    for i, f in enumerate(findings, 1):
        findings_text += f"""
--- Document {i}: {f.get('doc_path', 'unknown')} ---
Findings: {f.get('findings', 'N/A')}
Key facts: {json.dumps(f.get('key_facts', []))}
"""

    connections_text = ""
    if connections and connections.get("connections"):
        for c in connections["connections"]:
            connections_text += f"""
- [{c.get('type', 'unknown')}] Between {', '.join(c.get('documents', []))}: {c.get('description', '')}
  Significance: {c.get('significance', 'N/A')}"""
    if connections and connections.get("synthesis"):
        connections_text += f"\n\nCross-document synthesis: {connections['synthesis']}"

    user_prompt = f"""User question: {user_query}

Document analysis findings:
{findings_text}

Cross-document connections:
{connections_text if connections_text else "No cross-document connections analyzed."}

Please compose a comprehensive answer to the user's question based on the above findings."""

    return llm_call(SYNTHESIZER_SYSTEM_PROMPT, user_prompt, conversation=conversation_history)


print("Answer Synthesizer sub-agent ready.")

## 4. Orchestrator Agent

The orchestrator operates in a plan-act-observe loop. It receives the user query and document manifest,
plans which documents to analyze, dispatches sub-agents, observes results, and iterates until it has
enough information to synthesize a final answer.

In [ ]:
# ---------------------------------------------------------------------------
# Orchestrator Agent
# ---------------------------------------------------------------------------

ORCHESTRATOR_SYSTEM_PROMPT = """You are an orchestrator agent. You plan and coordinate document analysis
to answer the user's question. You have access to a set of documents and can dispatch sub-agents.

Available actions (respond with JSON):
- {{"action": "summarize", "doc_path": "..."}} -- Get a summary of a document
- {{"action": "analyze", "doc_path": "...", "question": "..."}} -- Analyze a document with a specific question
- {{"action": "find_connections"}} -- Find connections across all findings so far (use after 2+ analyses)
- {{"action": "synthesize"}} -- Compile final answer (use when you have enough information)

Always respond with ONLY valid JSON in this format:
{{"reasoning": "your step-by-step thinking about what to do next", "actions": [...]}}

Guidelines:
- Start by analyzing the most promising documents based on their names/types and the user's question
- After initial analysis, look for connections between findings
- Do not analyze more than 5-6 documents unless the query truly requires breadth
- Always call find_connections before synthesize if you analyzed 2+ documents
- Call synthesize when you have sufficient information to answer the question
- You can request multiple actions per step (they run sequentially)"""


def orchestrator_run(user_query, conversation_history=None, status_callback=None):
    """Run the orchestrator agent loop to answer a user query."""

    def update_status(msg):
        if status_callback:
            status_callback(msg)

    execution_log = []
    all_findings = []
    all_connections = None
    documents_analyzed = set()

    for iteration in range(MAX_ITERATIONS):
        update_status(f"Planning (iteration {iteration + 1}/{MAX_ITERATIONS})...")

        # Build execution log summary for the orchestrator
        log_summary = ""
        if execution_log:
            for entry in execution_log:
                log_summary += f"\n[Step {entry['step']}] Action: {entry['action']}"
                if entry['action'] == 'summarize':
                    log_summary += f" | Doc: {entry['input'].get('doc_path', '?')}"
                    log_summary += f"\n  Result: {str(entry['result'])[:500]}"
                elif entry['action'] == 'analyze':
                    log_summary += f" | Doc: {entry['input'].get('doc_path', '?')} | Q: {entry['input'].get('question', '?')}"
                    log_summary += f"\n  Findings: {str(entry['result'].get('findings', ''))[:500]}"
                    log_summary += f"\n  Entities: {entry['result'].get('key_entities', [])}"
                    log_summary += f"\n  Facts: {str(entry['result'].get('key_facts', []))[:300]}"
                elif entry['action'] == 'find_connections':
                    conns = entry['result'].get('connections', [])
                    log_summary += f"\n  Found {len(conns)} connections"
                    log_summary += f"\n  Synthesis: {str(entry['result'].get('synthesis', ''))[:500]}"

        user_prompt = f"""User query: {user_query}

Available documents:
{document_manifest}

{'Execution log so far:' + log_summary if log_summary else 'No actions taken yet.'}

What should we do next? Respond with JSON."""

        if conversation_history:
            conv_context = "\n".join(
                f"{'User' if m['role'] == 'user' else 'Assistant'}: {m['content'][:200]}"
                for m in conversation_history[-6:]  # last 3 turns
            )
            user_prompt = f"Recent conversation:\n{conv_context}\n\n{user_prompt}"

        raw = llm_call(ORCHESTRATOR_SYSTEM_PROMPT, user_prompt, llm_id=AGENT_LLM_ID)

        try:
            plan = parse_agent_json(raw)
        except (json.JSONDecodeError, ValueError):
            # If orchestrator fails to produce JSON, force synthesis
            plan = {"reasoning": "Failed to parse plan, synthesizing with available findings.", "actions": [{"action": "synthesize"}]}

        actions = plan.get("actions", [])
        if not actions:
            actions = [{"action": "synthesize"}]

        should_synthesize = False

        for action in actions:
            act_type = action.get("action", "")
            step_num = len(execution_log) + 1

            if act_type == "summarize":
                doc_path = action.get("doc_path", "")
                update_status(f"Summarizing {doc_path}...")
                result = agent_summarize_document(doc_path)
                execution_log.append({
                    "step": step_num,
                    "action": "summarize",
                    "input": {"doc_path": doc_path},
                    "result": {"summary": result},
                })

            elif act_type == "analyze":
                doc_path = action.get("doc_path", "")
                question = action.get("question", user_query)
                update_status(f"Analyzing {doc_path}...")
                result = agent_analyze_document(doc_path, question)
                all_findings.append(result)
                documents_analyzed.add(doc_path)
                execution_log.append({
                    "step": step_num,
                    "action": "analyze",
                    "input": {"doc_path": doc_path, "question": question},
                    "result": result,
                })

            elif act_type == "find_connections":
                if len(all_findings) >= 2:
                    update_status("Finding cross-document connections...")
                    all_connections = agent_find_connections(all_findings, user_query)
                    execution_log.append({
                        "step": step_num,
                        "action": "find_connections",
                        "input": {"num_findings": len(all_findings)},
                        "result": all_connections,
                    })

            elif act_type == "synthesize":
                should_synthesize = True
                break

        if should_synthesize:
            break

    # Final synthesis (either requested by orchestrator or forced at max iterations)
    update_status("Synthesizing final answer...")

    # If we have 2+ findings but never found connections, do it now
    if len(all_findings) >= 2 and all_connections is None:
        all_connections = agent_find_connections(all_findings, user_query)

    answer = agent_synthesize_answer(user_query, all_findings, all_connections, conversation_history)

    return {
        "answer": answer,
        "execution_log": execution_log,
        "iterations": iteration + 1,
        "documents_analyzed": list(documents_analyzed),
        "connections_found": all_connections.get("connections", []) if all_connections else [],
    }


print("Orchestrator agent ready.")

## 5. Interactive Chatbot Interface

A chatbot UI with an expandable agent trace showing the orchestrator's reasoning, sub-agent dispatches, and cross-document connections.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ---------------------------------------------------------------------------
# Chat state
# ---------------------------------------------------------------------------
conversation_history = []
chat_log_entries = []  # {"role": str, "content": str, "trace": dict|None}

# ---------------------------------------------------------------------------
# UI components
# ---------------------------------------------------------------------------
chat_output = widgets.Output(layout=widgets.Layout(
    width="100%",
    min_height="300px",
    max_height="600px",
    overflow_y="auto",
    border="1px solid #ccc",
    padding="10px",
))

user_input = widgets.Text(
    placeholder="Ask a question about your documents...",
    layout=widgets.Layout(width="80%"),
)

send_button = widgets.Button(
    description="Send",
    button_style="primary",
    layout=widgets.Layout(width="10%"),
)

clear_button = widgets.Button(
    description="Clear",
    button_style="warning",
    layout=widgets.Layout(width="10%"),
)

status_label = widgets.Label(value="Ready.")


def format_agent_trace(result):
    """Format the orchestrator execution trace as an HTML details block."""
    log = result.get("execution_log", [])
    iterations = result.get("iterations", 0)
    docs = result.get("documents_analyzed", [])
    connections = result.get("connections_found", [])

    trace_items = f"<b>Iterations:</b> {iterations}<br>"
    trace_items += f"<b>Documents analyzed:</b> {len(docs)}<br>"

    if docs:
        trace_items += "<ul>" + "".join(f"<li>{d}</li>" for d in docs) + "</ul>"

    if log:
        trace_items += "<b>Execution steps:</b><ol>"
        for entry in log:
            action = entry["action"]
            if action == "summarize":
                trace_items += f"<li>Summarized <code>{entry['input'].get('doc_path', '?')}</code></li>"
            elif action == "analyze":
                trace_items += (
                    f"<li>Analyzed <code>{entry['input'].get('doc_path', '?')}</code>"
                    f" &mdash; Q: <em>{entry['input'].get('question', '?')[:100]}</em></li>"
                )
            elif action == "find_connections":
                num = len(entry["result"].get("connections", []))
                trace_items += f"<li>Found {num} cross-document connections</li>"
        trace_items += "</ol>"

    if connections:
        trace_items += "<b>Connections discovered:</b><ul>"
        for c in connections:
            trace_items += (
                f"<li>[{c.get('type', '?')}] {c.get('description', '')[:150]}"
                f" <em>({', '.join(c.get('documents', []))})</em></li>"
            )
        trace_items += "</ul>"

    return (
        f'<details style="margin-top:6px; font-size:0.85em; color:#666;">'
        f'<summary>Agent Trace ({len(log)} steps, {len(connections)} connections)</summary>'
        f'{trace_items}</details>'
    )


def render_chat():
    """Re-render the chat output area."""
    with chat_output:
        clear_output(wait=True)
        for entry in chat_log_entries:
            if entry["role"] == "user":
                display(HTML(
                    f'<div style="margin:8px 0; padding:8px 12px; '
                    f'background:#e3f2fd; border-radius:8px;">'
                    f'<b>You:</b> {entry["content"]}</div>'
                ))
            else:
                trace_html = ""
                if entry.get("trace"):
                    trace_html = format_agent_trace(entry["trace"])
                answer_html = entry["content"].replace("\n", "<br>")
                display(HTML(
                    f'<div style="margin:8px 0; padding:8px 12px; '
                    f'background:#f1f8e9; border-radius:8px;">'
                    f'<b>Assistant:</b><br>{answer_html}{trace_html}</div>'
                ))


def on_send(button=None):
    """Handle send button click or Enter key."""
    question = user_input.value.strip()
    if not question:
        return

    user_input.value = ""
    chat_log_entries.append({"role": "user", "content": question, "trace": None})
    render_chat()

    def status_cb(msg):
        status_label.value = msg

    status_label.value = "Thinking..."
    try:
        result = orchestrator_run(
            question,
            conversation_history=conversation_history,
            status_callback=status_cb,
        )
        answer = result["answer"]

        conversation_history.append({"role": "user", "content": question})
        conversation_history.append({"role": "assistant", "content": answer})

        chat_log_entries.append({"role": "assistant", "content": answer, "trace": result})
        status_label.value = "Ready."
    except Exception as e:
        chat_log_entries.append({"role": "assistant", "content": f"Error: {e}", "trace": None})
        status_label.value = f"Error: {e}"

    render_chat()


def on_clear(button=None):
    """Clear the conversation."""
    conversation_history.clear()
    chat_log_entries.clear()
    # Clear cached summaries
    for doc in documents:
        doc["summary"] = None
    render_chat()
    status_label.value = "Conversation cleared."


# Wire up events
send_button.on_click(on_send)
user_input.on_submit(on_send)
clear_button.on_click(on_clear)

# Layout
input_row = widgets.HBox([user_input, send_button, clear_button])
chat_ui = widgets.VBox([chat_output, input_row, status_label])

display(HTML(
    f"<p><b>Config:</b> Orchestrator={AGENT_LLM_ID} | Sub-agents={SUB_AGENT_LLM_ID} | "
    f"Documents={len(documents)} | Max iterations={MAX_ITERATIONS}</p>"
))
display(chat_ui)